# Scanning Simulation Outputs

Tools for checking completion status across the `iz*/ivol*` directory tree and pooling all data from a snapshot.

In [1]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np

import galform_analysis
from galform_analysis import set_base_dir, get_base_dir, SimulationConfig, load_redshift_mapping

mpl.setconfig()

# Configure the path to your GALFORM output directory once per session:
# set_base_dir('/cosma5/data/durham/<user>/Galform_Out/L800/model')
BASE_DIR = str(get_base_dir())
SIM = 'L800'

## Completion summary

In [2]:
from galform_analysis.analysis.aggregation import completed_galaxies, incomplete_subvolumes
import polars as pl

df = completed_galaxies(basedir=BASE_DIR, iz_snapshots=[155, 207])
summary = (
    df.group_by('iz')
    .agg(
        pl.col('completed').sum().alias('n_complete'),
        pl.len().alias('n_total'),
    )
    .sort('iz')
)
print(summary)

shape: (2, 3)
┌───────┬────────────┬─────────┐
│ iz    ┆ n_complete ┆ n_total │
│ ---   ┆ ---        ┆ ---     │
│ str   ┆ u32        ┆ u32     │
╞═══════╪════════════╪═════════╡
│ iz155 ┆ 1024       ┆ 1024    │
│ iz207 ┆ 1024       ┆ 1024    │
└───────┴────────────┴─────────┘


## Missing or corrupted subvolumes

In [3]:
missing = incomplete_subvolumes(basedir=BASE_DIR, iz_snapshots=[155])
print(f"Incomplete subvolumes in iz155: {len(missing)}")
if len(missing):
    print(missing.head())

Incomplete subvolumes in iz155: 0


## Aggregate all subvolumes in a snapshot

In [ ]:
from galform_analysis.analysis.aggregation import aggregate_snapshot
import os

agg = aggregate_snapshot(os.path.join(BASE_DIR, 'iz155'))
print(f"Snapshot  : {agg['iz']},  z={agg['z']:.4f}")
print(f"Galaxies  : {len(agg['mstar']):,}")
print(f"Total vol : {agg['volume']:.1f} (Mpc/h)^3")

# Use agg['mstar'] / agg['mhalo'] directly for mass-function calculations
mstar = agg['mstar']
fig, ax = plt.subplots()
ax.hist(np.log10(mstar[mstar > 0]), bins=50, histtype='step', lw=1.5)
ax.set_xlabel(r'$\log_{10}(M_\star\,[\mathrm{M}_\odot/h])$')
ax.set_ylabel('Count')
ax.set_title(f"All galaxies in {agg['iz']}  ({len(agg['mstar']):,} total)")
fig.tight_layout()
plt.show()